# Mammogram Data Preprocessing & Probability Score Generation

In [ ]:
import os
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, classification_report

print('Libraries loaded ✓')

## load Raw CSVs

- `mass_case_description_train_set.csv`
- `mass_case_description_test_set.csv`
- `calc_case_description_train_set.csv`
- `calc_case_description_test_set.csv`
- `dicom_info.csv` 
  

In [ ]:
BASE_PATH       = '/kaggle/input/datasets/awsaf49/cbis-ddsm-breast-cancer-image-dataset'
MASSES_TRAIN    = os.path.join(BASE_PATH, 'csv/mass_case_description_train_set.csv')
MASSES_TEST     = os.path.join(BASE_PATH, 'csv/mass_case_description_test_set.csv')
CALC_TRAIN      = os.path.join(BASE_PATH, 'csv/calc_case_description_train_set.csv')
CALC_TEST       = os.path.join(BASE_PATH, 'csv/calc_case_description_test_set.csv')
DICOM_CSV       = os.path.join(BASE_PATH, 'csv/dicom_info.csv')       # لو مش zip
OUTPUT_DIR      = '/kaggle/working'

mass_train = pd.read_csv(MASSES_TRAIN)
mass_test  = pd.read_csv(MASSES_TEST)
calc_train = pd.read_csv(CALC_TRAIN)
calc_test  = pd.read_csv(CALC_TEST)
dicom = pd.read_csv(DICOM_CSV)

print('Files loaded:')
for name, df in [('mass_train', mass_train), ('mass_test', mass_test),
                 ('calc_train', calc_train), ('calc_test',  calc_test),
                 ('dicom',      dicom)]:
    print(f'  {name:<12s}  shape={df.shape}')

In [ ]:
def prep_mammo(df, split, abn_type):
    df = df.copy()
    df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')
   
    df['split']            = split
    df['abnormality_type'] = abn_type

    for col in ['image_file_path', 'cropped_image_file_path', 'roi_mask_file_path']:
        if col in df.columns:
            df[col] = df[col].str.strip()

    df.rename(columns={'breast density': 'breast_density'}, inplace=True)
    return df

mass_train_c = prep_mammo(mass_train, 'train', 'mass')
mass_test_c  = prep_mammo(mass_test,  'test',  'mass')
calc_train_c = prep_mammo(calc_train, 'train', 'calcification')
calc_test_c  = prep_mammo(calc_test,  'test',  'calcification')

mammo = pd.concat(
    [mass_train_c, mass_test_c, calc_train_c, calc_test_c],
    ignore_index=True
)

print(f'Combined shape: {mammo.shape}')
print(f'Columns: {mammo.columns.tolist()}')
print()
print('split dist:           ', mammo['split'].value_counts().to_dict())
print('abnormality_type dist:', mammo['abnormality_type'].value_counts().to_dict())
print('pathology dist:       ', mammo['pathology'].value_counts().to_dict())

In [ ]:
# ── Build dicom lookups ───────────────────────────────────────────────────────
dicom_full = (
    dicom[dicom['SeriesDescription'] == 'full mammogram images']
    [['PatientID', 'image_path']]
    .rename(columns={'PatientID': 'key_full', 'image_path': 'full_img_path'})
)
dicom_crop = (
    dicom[dicom['SeriesDescription'] == 'cropped images']
    [['PatientID', 'image_path']]
    .rename(columns={'PatientID': 'key_cropped', 'image_path': 'cropped_img_path'})
)
dicom_roi = (
    dicom[dicom['SeriesDescription'] == 'ROI mask images']
    [['PatientID', 'image_path']]
    .rename(columns={'PatientID': 'key_roi', 'image_path': 'roi_img_path'})
)

print('Dicom lookup sizes:')
print(f'  full:    {len(dicom_full)}')
print(f'  cropped: {len(dicom_crop)}')
print(f'  roi:     {len(dicom_roi)}')

# ── Extract join keys from path columns ──────────────────────────────────────
# الـ folder name في الـ image_file_path = الـ PatientID في dicom
# مثال: 'Mass-Training_P_01265_RIGHT_MLO_1/...' → 'Mass-Training_P_01265_RIGHT_MLO_1'
mammo['key_full']    = mammo['image_file_path'].str.split('/').str[0]
mammo['key_cropped'] = mammo['cropped_image_file_path'].str.split('/').str[0]
mammo['key_roi']     = mammo['roi_mask_file_path'].str.split('/').str[0]

# ── Merge ─────────────────────────────────────────────────────────────────────
mammo = mammo.merge(dicom_full, on='key_full',    how='left')
mammo = mammo.merge(dicom_crop, on='key_cropped', how='left')
mammo = mammo.merge(dicom_roi,  on='key_roi',     how='left')

# ── Check match rates ─────────────────────────────────────────────────────────
print()
print('Match rates after merge:')
print(f'  full_img_path:    {mammo["full_img_path"].notnull().sum()}/{len(mammo)} present')
print(f'  cropped_img_path: {mammo["cropped_img_path"].notnull().sum()}/{len(mammo)} present')
print(f'  roi_img_path:     {mammo["roi_img_path"].notnull().sum()}/{len(mammo)} present')
print()


In [ ]:
# ── Binary label ──────────────────────────────────────────────────────────────
mammo['label'] = mammo['pathology'].map({
    'MALIGNANT':              1,
    'BENIGN':                 0,
    'BENIGN_WITHOUT_CALLBACK': 0
})

# ── Drop helper columns ───────────────────────────────────────────────────────
drop_cols = [
    'image_file_path', 'cropped_image_file_path', 'roi_mask_file_path',
    'key_full', 'key_cropped', 'key_roi'
]
mammo.drop(columns=drop_cols, inplace=True)

# ── Final column order ────────────────────────────────────────────────────────
col_order = [
    'patient_id', 'split', 'abnormality_type', 'breast_density',
    'left_or_right_breast', 'image_view', 'abnormality_id',
    'mass_shape', 'mass_margins',
    'calc_type', 'calc_distribution',
    'assessment', 'subtlety', 'pathology', 'label',
    'full_img_path', 'cropped_img_path', 'roi_img_path'
]
mammo = mammo[col_order]

# ── Save merged table ─────────────────────────────────────────────────────────
merged_path = os.path.join(OUTPUT_DIR, 'mammogram_merged.csv')
mammo.to_csv(merged_path, index=False)

print(f'Saved → {merged_path}')
print(f'Shape: {mammo.shape}')
print()
print(mammo.head(3).to_string())

## 4. تنضيف الداتا (Preprocessing)

**اللي بنعمله:**
1. شيل الصفوف اللي مفيهاش `full_img_path` (الـ 282 Calc-Test)
2. شيل الـ columns مش مفيدة للـ tabular model
3. Handle الـ nulls:
   - `calc_type/calc_distribution` للـ mass rows → `NONE` (by design مش missing)
   - `mass_shape/mass_margins` للـ calc rows → `NONE` (by design مش missing)
   - الـ nulls الحقيقية الصغيرة → mode imputation
4. One-hot encoding للـ categorical columns

In [ ]:
# ── Step 1: drop rows without full_img_path ────────────────────────────
print(f'Before filter: {len(mammo)} rows')
df = mammo[mammo['full_img_path'].notnull()].copy()
print(f'After filter:  {len(df)} rows  (removed {len(mammo)-len(df)} rows without full_img_path)')
print()

# ── Step 2: drop columns of tabular model ──────────────────────
drop_for_model = [
    'patient_id',           
    'abnormality_id',       
    'pathology',           
    'image_view',           
    'left_or_right_breast',
    'cropped_img_path', 
    'assessment',
    'subtlety',
    'roi_img_path',
    
]
df.drop(columns=drop_for_model, inplace=True)
print('Columns kept for model:')
print(' ', [c for c in df.columns if c not in ['label', 'split', 'full_img_path']])
print()

# ── Step 3: Handle nulls ─────────────────────────────────────────────────────
# calc_type / calc_distribution → NONE للـ mass rows (by design)
# mass_shape / mass_margins    → NONE للـ calc rows (by design)
for col in ['calc_type', 'calc_distribution', 'mass_shape', 'mass_margins']:
    df[col] = df[col].fillna('NONE')

for abn in ['mass', 'calcification']:
    mask = df['abnormality_type'] == abn
    own_cols = (
        ['mass_shape', 'mass_margins']
        if abn == 'mass'
        else ['calc_type', 'calc_distribution']
    )
    for col in own_cols:
        real_missing = mask & (df[col] == 'NONE')
        if real_missing.sum() > 0:
            mode_val = df.loc[mask & (df[col] != 'NONE'), col].mode()[0]
            df.loc[real_missing, col] = mode_val
            print(f'  Filled {real_missing.sum()} real nulls in {col} ({abn}) with "{mode_val}"')

print()
print('Nulls after cleaning:')
print(df.isnull().sum()[df.isnull().sum() > 0])
if df.isnull().sum().sum() == 0:
    print('  No nulls remaining ✓')

In [ ]:
import pandas as pd
from catboost import CatBoostClassifier
from sklearn.metrics import roc_auc_score, classification_report

df_features = df.drop(columns=['label', 'split', 'pathology'], errors='ignore')

df_features = df_features.drop_duplicates(subset=['full_img_path'], keep='first')

train_split = pd.read_csv('/kaggle/input/datasets/habibashhefny/mammo-ps-split/mammo_ft_train.csv')
val_split   = pd.read_csv('/kaggle/input/datasets/habibashhefny/mammo-ps-split/mammo_ft_val.csv')
test_split  = pd.read_csv('/kaggle/input/datasets/habibashhefny/mammo-ps-split/mammo_final_test.csv')

train_tab = pd.merge(train_split[['full_img_path', 'label']], df_features, on='full_img_path', how='inner')
val_tab   = pd.merge(val_split[['full_img_path', 'label']], df_features, on='full_img_path', how='inner')
test_tab  = pd.merge(test_split[['full_img_path', 'label']], df_features, on='full_img_path', how='inner')

print(f"Tabular Train shape: {train_tab.shape}") 
print(f"Tabular Val shape:   {val_tab.shape}")   
print(f"Tabular Test shape:  {test_tab.shape}")  

cat_cols = ['abnormality_type', 'breast_density', 'mass_shape', 'mass_margins', 'calc_type', 'calc_distribution']

for col in cat_cols:
    if col in train_tab.columns:
        train_tab[col] = train_tab[col].fillna('MISSING').astype(str)
        val_tab[col]   = val_tab[col].fillna('MISSING').astype(str)
        test_tab[col]  = test_tab[col].fillna('MISSING').astype(str)

X_train = train_tab[cat_cols]
y_train = train_tab['label']

X_val = val_tab[cat_cols]
y_val = val_tab['label']

X_test = test_tab[cat_cols]
y_test = test_tab['label']

print("\nTraining CatBoost strictly on Train split...")
model = CatBoostClassifier(
    iterations=500, 
    learning_rate=0.05, 
    depth=6, 
    cat_features=cat_cols, 
    verbose=100, 
    auto_class_weights='Balanced' 
)
model.fit(X_train, y_train, eval_set=(X_val, y_val), early_stopping_rounds=50)

train_tab['PS_tabular'] = model.predict_proba(X_train)[:, 1]
val_tab['PS_tabular']   = model.predict_proba(X_val)[:, 1]
test_tab['PS_tabular']  = model.predict_proba(X_test)[:, 1]

test_preds = (test_tab['PS_tabular'] >= 0.5).astype(int)
print("\nTabular Model Performance on UNSEEN Final Test:")
print(f"AUC: {roc_auc_score(y_test, test_tab['PS_tabular']):.4f}")
print(classification_report(y_test, test_preds))

train_tab[['full_img_path', 'label', 'PS_tabular']].to_csv('/kaggle/working/tabular_ft_train.csv', index=False)
val_tab[['full_img_path', 'label', 'PS_tabular']].to_csv('/kaggle/working/tabular_ft_val.csv', index=False)
test_tab[['full_img_path', 'label', 'PS_tabular']].to_csv('/kaggle/working/tabular_final_test.csv', index=False)

print("\n 3csvs are saved")

In [ ]:
from sklearn.metrics import roc_auc_score, classification_report

train_preds = (train_tab['PS_tabular'] >= 0.5).astype(int)
print("📊 Tabular Model Performance on TRAIN Set:")
print(f"AUC: {roc_auc_score(y_train, train_tab['PS_tabular']):.4f}")
print(classification_report(y_train, train_preds))
print("=" * 55)

val_preds = (val_tab['PS_tabular'] >= 0.5).astype(int)
print("📊 Tabular Model Performance on VAL Set:")
print(f"AUC: {roc_auc_score(y_val, val_tab['PS_tabular']):.4f}")
print(classification_report(y_val, val_preds))
print("=" * 55)